# Введение в MapReduce модель на Python


In [1]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [2]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)
    
def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [3]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [4]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [5]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [6]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [7]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [8]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [9]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [10]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [11]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [12]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных. 

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [13]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*
 
mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*
 
mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL 

In [14]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str
    
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)
    
def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)
 
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication 

In [15]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])
 
def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])
      
output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(2.1205125123549227)),
 (1, np.float64(2.1205125123549227)),
 (2, np.float64(2.1205125123549227)),
 (3, np.float64(2.1205125123549227)),
 (4, np.float64(2.1205125123549227))]

## Inverted index 

In [16]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)
      
def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)
 
def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('is', ['0', '1', '2']),
 ('it', ['0', '1', '2']),
 ('what', ['0', '1']),
 ('banana', ['2']),
 ('a', ['2'])]

## WordCount

In [17]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):  
    yield (word, 1)
 
def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [18]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()
      
def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]
 
def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers
  
def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)
  
  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*
 
e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*
 
flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount 

In [19]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps
  
  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)
      
  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):  
    yield (word, 1)
 
def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)
  
# try to set COMBINER=REDUCER and look at the number of values sent over the network 
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None) 
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('a', 2), ('banana', 2), ('is', 18), ('it', 18), ('what', 10)]),
 (1, [])]

## TeraSort

In [20]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps
  
  def RECORDREADER(split):
    for value in split:
        yield (value, None)
      
  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])
    
def MAP(value:int, _):
  yield (value, None)
  
def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)
  
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.09283554775227487)),
   (None, np.float64(0.13085462066191067)),
   (None, np.float64(0.21638683574014395)),
   (None, np.float64(0.3130114881605661)),
   (None, np.float64(0.3176322022906446)),
   (None, np.float64(0.33728960997560253)),
   (None, np.float64(0.34986901969108564)),
   (None, np.float64(0.3575872107030488)),
   (None, np.float64(0.43677666341446475)),
   (None, np.float64(0.4663930251428331)),
   (None, np.float64(0.4690523323894923)),
   (None, np.float64(0.47442467672957256)),
   (None, np.float64(0.4886349268640925))]),
 (1,
  [(None, np.float64(0.5072522291327407)),
   (None, np.float64(0.5115895039841588)),
   (None, np.float64(0.5295481627976603)),
   (None, np.float64(0.5956758409524159)),
   (None, np.float64(0.5968965416585578)),
   (None, np.float64(0.5981143045852186)),
   (None, np.float64(0.6009068142905525)),
   (None, np.float64(0.6044925285146102)),
   (None, np.float64(0.6449996344517743)),
   (None, np.float64(0.780139119883

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [22]:
def RECORDREADER():
    data = [1, 452, 233, 22, 76, 9, 44]
    for i, value in enumerate(data):
        yield (i, value)

def MAP(_, value):
    yield ('max', value)

def REDUCE(key, values):
    yield ('max', max(values))

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(output[0][1])

452


### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [24]:
def RECORDREADER():
    data = [1, 2, 3, 4, 5, 6, 7, 8, 9]
    for i, value in enumerate(data):
        yield (i, value)

def MAP(_, value):
    yield ('mean', (value, 1))

def REDUCE(key, values):
    total_sum = sum(v[0] for v in values)
    total_count = sum(v[1] for v in values)
    mean = total_sum / total_count if total_count > 0 else 0
    yield ('mean', mean)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(output[0][1])

5.0


### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [25]:
def groupbykey_sorted(iterable):
    sorted_items = sorted(iterable, key=lambda x: x[0])
    groups = {}
    for key, value in sorted_items:
        if key not in groups:
            groups[key] = []
        groups[key].append(value)
    return groups.items()

test_data = [('f', 1), ('b', 4), ('a', 10), ('a', 15), ('e', 5)]
result = list(groupbykey_sorted(test_data))
print(result)

[('a', [10, 15]), ('b', [4]), ('e', [5]), ('f', [1])]


### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [26]:
def RECORDREADER():
    data = [1, 2, 3, 4, 5, 6, 7, 8, 9, 1, 2, 3, 4, 5, 6, 7, 8, 9]
    for i, value in enumerate(data):
        yield (i, value)

def MAP(_, value):
    yield (value, None)

def REDUCE(key, values):
    yield (key, None)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
unique_values = sorted([k for k, v in output])
print(unique_values)

[1, 2, 3, 4, 5, 6, 7, 8, 9]


#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [27]:
from collections import defaultdict
import random

def map_operation(data_batch):
    intermediate = defaultdict(list)
    for record in data_batch:
        if filter_condition(record):
            intermediate[record].append(record)
    return intermediate.items()

def reduce_operation(mapped_data):
    output = []
    for item_group in mapped_data:
        for item in item_group:
            output.append(item)
    return output

def filter_condition(tuple_record):
    return tuple_record[0] % 2 == 0

def data_generator(num_records):
    return [(random.randint(0, 100), random.randint(0, 100), random.randint(0, 100)) for _ in range(num_records)]

dataset = data_generator(5)
print(dataset)

partition_size = 5
partitions = [dataset[i:i + partition_size] for i in range(0, len(dataset), partition_size)]
print(partitions)

mapped_partitions = list(map(lambda x: map_operation(x), partitions))
final_result = reduce_operation(mapped_partitions)
print(final_result)


[(9, 49, 50), (43, 59, 39), (51, 8, 93), (96, 82, 36), (28, 54, 23)]
[[(9, 49, 50), (43, 59, 39), (51, 8, 93), (96, 82, 36), (28, 54, 23)]]
[((96, 82, 36), [(96, 82, 36)]), ((28, 54, 23), [(28, 54, 23)])]


### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [34]:
filter_set = {44, 90, 2}

def map_fn(input_tuple):
    filtered = []
    for item in input_tuple:
        if item in filter_set:
            filtered.append(item)
    result = tuple(filtered)
    return (result, result)

def reduce_fn(key, values):
    return (key, key)

def data_source(num_records):
    return [(random.randint(0, 100), random.randint(0, 100), random.randint(0, 100)) for _ in range(num_records)]

def groupByKey(data):
    groups = {}
    for k, v in data:
        groups[k] = groups.get(k, []) + [v]
    return groups.items()

records = data_source(5)
print(records)

mapped = list(map(lambda x: map_fn(x), data_source(100)))
print("MAP output:", mapped)

shuffled = list(groupByKey(mapped))
print("Shuffle output:", shuffled)

reduced = list(map(lambda x: reduce_fn(*x)[0], shuffled))
print("Reduce output:", reduced)

[(42, 82, 59), (57, 28, 49), (75, 53, 3), (96, 39, 58), (66, 58, 41)]
MAP output: [((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((90,), (90,)), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((44,), (44,)), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((90,), (90,)), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((2,), (2,)), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((), ()), ((90,), (90,)), ((), ()), ((), ()), ((), ()), ((2,), (2,)), ((), ()), ((44,), (44,)), ((2,), (2,)), ((), ()), ((), ()), ((), 

### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [36]:
def map_fn(record):
    return (record, record)

def reduce_fn(key, values):
    return (key, key)

def data_source(num_records):
    return [(random.randint(0, 100), random.randint(0, 100), random.randint(0, 100)) for _ in range(num_records)]

def groupByKey(data):
    groups = {}
    for k, v in data:
        groups[k] = groups.get(k, []) + [v]
    return groups.items()

dataset = data_source(5)
print(dataset)

mapped = list(map(lambda x: map_fn(x), data_source(100)))
print("MAP output:", mapped)

shuffled = list(groupByKey(mapped))
print("Shuffle output:", shuffled)

reduced = list(map(lambda x: reduce_fn(*x)[0], shuffled))
print("Reduce output:", reduced)

[(43, 14, 9), (5, 36, 99), (79, 70, 87), (63, 52, 54), (76, 82, 91)]
MAP output: [((14, 98, 90), (14, 98, 90)), ((61, 5, 47), (61, 5, 47)), ((64, 6, 11), (64, 6, 11)), ((26, 61, 36), (26, 61, 36)), ((32, 15, 51), (32, 15, 51)), ((25, 91, 60), (25, 91, 60)), ((13, 48, 64), (13, 48, 64)), ((17, 22, 81), (17, 22, 81)), ((64, 38, 73), (64, 38, 73)), ((31, 29, 91), (31, 29, 91)), ((77, 59, 11), (77, 59, 11)), ((58, 24, 14), (58, 24, 14)), ((16, 78, 39), (16, 78, 39)), ((42, 4, 77), (42, 4, 77)), ((45, 93, 54), (45, 93, 54)), ((7, 18, 31), (7, 18, 31)), ((93, 63, 25), (93, 63, 25)), ((96, 46, 56), (96, 46, 56)), ((67, 33, 34), (67, 33, 34)), ((56, 26, 71), (56, 26, 71)), ((89, 50, 83), (89, 50, 83)), ((3, 77, 14), (3, 77, 14)), ((94, 81, 95), (94, 81, 95)), ((97, 54, 22), (97, 54, 22)), ((95, 33, 70), (95, 33, 70)), ((66, 28, 97), (66, 28, 97)), ((60, 53, 84), (60, 53, 84)), ((6, 58, 75), (6, 58, 75)), ((7, 35, 55), (7, 35, 55)), ((29, 51, 19), (29, 51, 19)), ((72, 57, 93), (72, 57, 93)), ((

### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [37]:
def map_fn(record):
    return (record, record)

def reduce_fn(key, values):
    if len(values) == 2:
        return (key, key)

def data_source(num_records):
    return [(random.randint(0, 3), random.randint(0, 3)) for _ in range(num_records)]

def groupByKey(data):
    groups = {}
    for k, v in data:
        groups[k] = groups.get(k, []) + [v]
    return groups.items()

dataset = data_source(100)
print(dataset)

mapped = list(map(lambda x: map_fn(x), data_source(100)))
print("MAP output:", mapped)

shuffled = list(groupByKey(mapped))
print("Shuffle output:", shuffled)

reduced = [item[0] for item in list(map(lambda x: reduce_fn(*x), shuffled)) if item is not None]
print("Reduce output:", reduced)

[(3, 1), (3, 2), (0, 2), (3, 2), (2, 1), (2, 1), (2, 3), (0, 3), (2, 2), (1, 0), (1, 0), (3, 2), (2, 1), (1, 3), (3, 3), (3, 3), (3, 2), (2, 2), (0, 1), (2, 1), (3, 2), (3, 2), (1, 0), (0, 3), (0, 0), (0, 3), (3, 1), (3, 1), (0, 0), (1, 3), (1, 0), (3, 3), (3, 1), (1, 1), (0, 1), (0, 2), (3, 3), (3, 2), (0, 0), (2, 2), (3, 3), (1, 2), (2, 2), (1, 1), (3, 2), (3, 1), (0, 2), (2, 1), (2, 0), (3, 0), (2, 1), (3, 1), (1, 1), (1, 3), (0, 1), (0, 0), (1, 3), (2, 1), (2, 3), (1, 1), (3, 2), (0, 2), (1, 3), (2, 2), (3, 2), (1, 2), (0, 1), (3, 3), (2, 1), (0, 2), (2, 1), (1, 2), (2, 2), (1, 1), (1, 1), (2, 2), (2, 0), (3, 2), (3, 1), (2, 0), (3, 1), (3, 3), (3, 1), (3, 1), (0, 0), (0, 2), (3, 1), (2, 2), (0, 3), (3, 3), (2, 3), (2, 1), (2, 1), (1, 2), (0, 2), (3, 0), (1, 0), (2, 0), (1, 2), (0, 0)]
MAP output: [((2, 1), (2, 1)), ((2, 1), (2, 1)), ((2, 1), (2, 1)), ((2, 0), (2, 0)), ((2, 0), (2, 0)), ((0, 0), (0, 0)), ((2, 0), (2, 0)), ((3, 0), (3, 0)), ((1, 0), (1, 0)), ((3, 1), (3, 1)), ((2, 2

### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [38]:
relation_ids = [1, 2]

class DataTuple:
    def __init__(self, values: tuple, relation_id: int):
        self.values = values
        self.relation_id = relation_id

def generate_random_tuple(count):
    values = tuple([(random.randint(0, 3), random.randint(0, 3)) for _ in range(count)])
    relation_id = relation_ids[random.randint(0, len(relation_ids) - 1)]
    return DataTuple(values, relation_id)

def data_source(num_records):
    return [generate_random_tuple(3) for _ in range(num_records)]

def map_fn(tuple_obj: DataTuple):
    return (tuple_obj.values, tuple_obj.relation_id)

def reduce_fn(key, values):
    if values == [relation_ids[0]]:
        return (key, key)

def groupByKey(data):
    groups = {}
    for k, v in data:
        groups[k] = groups.get(k, []) + [v]
    return groups.items()

dataset = data_source(100)

mapped = list(map(lambda x: map_fn(x), data_source(100)))
print("MAP output:", mapped)

shuffled = list(groupByKey(mapped))
print("Shuffle output:", shuffled)

reduced = [item[0] for item in list(map(lambda x: reduce_fn(*x), shuffled)) if item is not None]
print("Reduce output:", reduced)

MAP output: [(((1, 1), (1, 0), (1, 2)), 1), (((1, 1), (0, 2), (2, 2)), 1), (((1, 0), (1, 1), (3, 1)), 1), (((2, 1), (1, 3), (3, 2)), 2), (((3, 3), (1, 1), (3, 1)), 2), (((0, 3), (3, 2), (1, 0)), 2), (((2, 3), (3, 1), (2, 1)), 1), (((2, 2), (0, 3), (1, 2)), 2), (((1, 3), (1, 1), (1, 2)), 1), (((1, 3), (2, 1), (0, 1)), 1), (((2, 3), (3, 0), (2, 2)), 1), (((3, 3), (3, 1), (2, 3)), 2), (((0, 1), (3, 3), (3, 3)), 1), (((3, 2), (3, 0), (2, 0)), 2), (((3, 3), (1, 2), (1, 2)), 1), (((1, 1), (0, 3), (2, 3)), 2), (((3, 3), (0, 1), (1, 2)), 2), (((1, 0), (1, 1), (3, 0)), 2), (((0, 3), (1, 2), (2, 1)), 2), (((1, 2), (3, 0), (1, 2)), 1), (((2, 1), (2, 2), (2, 3)), 1), (((2, 3), (1, 2), (0, 2)), 2), (((1, 0), (0, 0), (3, 3)), 2), (((0, 1), (3, 0), (0, 1)), 1), (((0, 1), (3, 2), (1, 1)), 2), (((2, 2), (0, 2), (1, 3)), 1), (((1, 3), (0, 0), (2, 0)), 2), (((0, 1), (1, 0), (3, 0)), 2), (((1, 1), (3, 3), (3, 1)), 2), (((1, 3), (3, 1), (1, 1)), 1), (((1, 0), (2, 1), (2, 1)), 1), (((1, 0), (1, 3), (0, 3)),

### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [39]:
relation_ids = [1, 2]

class DataRecord:
    def __init__(self, values: tuple, relation_id: int):
        self.values = values
        self.relation_id = relation_id

def generate_record():
    vals = (random.randint(0, 3), random.randint(0, 3))
    rid = relation_ids[random.randint(0, len(relation_ids) - 1)]
    return DataRecord(vals, rid)

def data_source(num_records):
    return [generate_record() for _ in range(num_records)]

def map_fn(record: DataRecord):
    if record.relation_id == relation_ids[0]:
        return (record.values[1], (record.relation_id, record.values[0]))
    else:
        return (record.values[0], (record.relation_id, record.values[1]))

def reduce_fn(key, values):
    result = []
    for v in values:
        result.append((v[0], key, v[1]))
    return result

def groupByKey(data):
    groups = {}
    for k, v in data:
        groups[k] = groups.get(k, []) + [v]
    return groups.items()

dataset = data_source(100)

mapped = list(map(lambda x: map_fn(x), data_source(100)))
print("MAP output:", mapped)

shuffled = list(groupByKey(mapped))
print("Shuffle output:", shuffled)

reduced = list(map(lambda x: reduce_fn(*x), shuffled))
print("Reduce output:", reduced)

MAP output: [(1, (2, 2)), (3, (2, 3)), (3, (2, 2)), (1, (1, 0)), (3, (2, 0)), (0, (1, 3)), (3, (1, 3)), (1, (1, 1)), (1, (1, 0)), (1, (1, 0)), (1, (1, 2)), (0, (1, 1)), (1, (1, 1)), (0, (2, 0)), (1, (1, 2)), (2, (1, 2)), (2, (2, 3)), (0, (2, 0)), (2, (2, 0)), (0, (1, 1)), (1, (1, 0)), (0, (2, 2)), (2, (2, 0)), (2, (2, 1)), (1, (2, 0)), (0, (1, 0)), (1, (1, 1)), (2, (2, 0)), (3, (2, 0)), (1, (1, 1)), (2, (1, 0)), (2, (2, 3)), (2, (2, 0)), (2, (2, 0)), (3, (2, 0)), (2, (1, 1)), (0, (1, 3)), (3, (2, 2)), (1, (2, 3)), (0, (2, 1)), (3, (1, 2)), (3, (2, 1)), (1, (2, 2)), (0, (1, 2)), (2, (2, 3)), (1, (2, 1)), (3, (1, 2)), (0, (1, 2)), (0, (2, 0)), (1, (2, 3)), (3, (2, 1)), (1, (2, 2)), (2, (2, 0)), (3, (1, 1)), (0, (2, 3)), (2, (1, 0)), (1, (2, 2)), (2, (2, 1)), (2, (1, 0)), (1, (1, 3)), (0, (2, 2)), (1, (2, 0)), (1, (2, 1)), (3, (1, 3)), (1, (2, 2)), (1, (2, 3)), (2, (1, 0)), (0, (1, 3)), (1, (2, 0)), (2, (2, 0)), (1, (1, 2)), (3, (1, 0)), (3, (1, 2)), (3, (2, 2)), (1, (1, 2)), (2, (2, 1)),

### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [40]:
def generate_random_triple():
    return (random.randint(0, 3), random.randint(0, 3), random.randint(0, 3))

def data_source(num_records):
    return [generate_random_triple() for _ in range(num_records)]

def map_fn(record):
    return (record[0], record[1])

def aggregate_fn(values):
    return sum(values)

def reduce_fn(key, values):
    result = aggregate_fn(values)
    return (key, result)

def groupByKey(data):
    groups = {}
    for k, v in data:
        groups[k] = groups.get(k, []) + [v]
    return groups.items()

dataset = data_source(100)

mapped = list(map(lambda x: map_fn(x), data_source(100)))
print("MAP output:", mapped)

shuffled = list(groupByKey(mapped))
print("Shuffle output:", shuffled)

reduced = list(map(lambda x: reduce_fn(*x), shuffled))
print("Reduce output:", reduced)

MAP output: [(1, 3), (1, 3), (0, 2), (1, 0), (3, 3), (1, 0), (0, 0), (0, 0), (1, 0), (2, 0), (3, 0), (0, 1), (2, 1), (3, 3), (3, 0), (1, 2), (0, 1), (2, 3), (3, 1), (0, 2), (3, 0), (1, 1), (0, 2), (3, 3), (3, 2), (0, 3), (0, 1), (0, 1), (2, 2), (1, 1), (0, 2), (1, 3), (0, 1), (2, 1), (1, 2), (0, 1), (1, 3), (0, 3), (1, 2), (2, 2), (1, 0), (0, 0), (2, 1), (2, 2), (3, 1), (2, 3), (0, 0), (0, 0), (1, 2), (1, 3), (3, 3), (2, 0), (0, 0), (1, 0), (3, 0), (1, 0), (1, 1), (2, 3), (3, 1), (3, 0), (1, 0), (0, 0), (0, 1), (0, 0), (0, 1), (1, 2), (0, 1), (0, 1), (2, 2), (0, 1), (3, 0), (0, 0), (3, 3), (3, 1), (0, 1), (0, 1), (3, 2), (3, 2), (3, 1), (2, 0), (3, 3), (2, 3), (0, 1), (0, 2), (3, 0), (2, 3), (0, 2), (1, 1), (3, 0), (2, 1), (3, 3), (1, 2), (1, 0), (1, 0), (1, 3), (0, 2), (3, 2), (0, 3), (3, 3), (0, 2)]
Shuffle output: [(1, [3, 3, 0, 0, 0, 2, 1, 1, 3, 2, 3, 2, 0, 2, 3, 0, 0, 1, 0, 2, 1, 2, 0, 0, 3]), (0, [2, 0, 0, 1, 1, 2, 2, 3, 1, 1, 2, 1, 1, 3, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1,

# 

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [41]:
from typing import Iterator, Tuple
import numpy as np

In [42]:
M, N = 5, 4
chunk_width = 2

matrix_data = np.random.rand(M, N)
vector_data = np.random.rand(N)

def data_stream() -> Iterator[Tuple[int, Tuple[int, int, float]]]:
    for chunk_idx in range((N + chunk_width - 1) // chunk_width):
        col_start = chunk_idx * chunk_width
        col_end = min(col_start + chunk_width, N)
        for row_idx in range(M):
            for col_idx in range(col_start, col_end):
                yield chunk_idx, (row_idx, col_idx, matrix_data[row_idx, col_idx])

def map_phase(chunk_idx: int, entry: Tuple[int, int, float]) -> Iterator[Tuple[int, float]]:
    row_idx, col_idx, val = entry
    chunk_vec = vector_data[chunk_idx*chunk_width : chunk_idx*chunk_width + chunk_width]
    result = val * chunk_vec[col_idx - chunk_idx*chunk_width]
    yield row_idx, result

def reduce_phase(row_idx: int, contributions: Iterator[float]) -> Iterator[Tuple[int, float]]:
    yield row_idx, sum(contributions)

def flatten(data):
    for item in data:
        for elem in item:
            yield elem

def groupbykey(items):
    groups = {}
    for k, v in items:
        groups.setdefault(k, []).append(v)
    return groups.items()

stream = list(data_stream())
mapped = list(flatten(map(lambda x: map_phase(*x), stream)))
shuffled = list(groupbykey(mapped))
final = list(flatten(map(lambda x: reduce_phase(*x), shuffled)))

computed = [v for _, v in final]
expected = np.matmul(matrix_data, vector_data)
print(np.allclose(computed, expected))

True


## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$. 





In [43]:
# MapReduce model
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [44]:
import numpy as np
I = 2
J = 3
K = 4*10
small_mat = np.random.rand(I,J) # it is legal to access this from RECORDREADER, MAP, REDUCE
big_mat = np.random.rand(J,K)

def RECORDREADER():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      yield ((j,k), big_mat[j,k])
      
def MAP(k1, v1):
  (j, k) = k1
  w = v1
  for i in range(small_mat.shape[0]):
    yield ((i, k), w * small_mat[i][j])

def REDUCE(key, values):
  (i, k) = key
  el_value = 0
  for v in values:
    el_value += v
  yield ((i, k), el_value)

Проверьте своё решение

In [45]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat) 
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

In [46]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [47]:
I, J, K = 4, 10, 30
mat_a = np.random.rand(I, J)
mat_b = np.random.rand(J, K)
expected = np.matmul(mat_a, mat_b)

def data_stream():
    for i in range(mat_a.shape[0]):
        for j in range(mat_a.shape[1]):
            yield ((0, i, j), mat_a[i, j])
    for j in range(mat_b.shape[0]):
        for k in range(mat_b.shape[1]):
            yield ((1, j, k), mat_b[j, k])

def map_join(key, val):
    src, idx1, idx2 = key
    if src == 0:
        yield (idx2, (src, idx1, val))
    else:
        yield (idx1, (src, idx2, val))

def reduce_join(key, vals):
    left = [v for v in vals if v[0] == 0]
    right = [v for v in vals if v[0] == 1]
    for l in left:
        for r in right:
            yield ((l[1], r[1]), l[2] * r[2])

def map_pass(key, val):
    yield (key, val)

def reduce_sum(key, vals):
    yield (key, sum(vals))

def stream_joined():
    for item in joined_result:
        yield item

joined_result = MapReduce(data_stream, map_join, reduce_join)
final_result = MapReduce(stream_joined, map_pass, reduce_sum)

def to_matrix(output):
    out_list = list(output)
    rows = max(i for ((i, k), _) in out_list) + 1
    cols = max(k for ((i, k), _) in out_list) + 1
    result_mat = np.empty((rows, cols))
    for (i_k, val) in out_list:
        result_mat[i_k] = val
    return result_mat

print(np.allclose(expected, to_matrix(final_result)))

True


Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER. 

In [48]:
rows_a, common_dim, cols_b = 4, 10, 30
matrix_a = np.random.rand(rows_a, common_dim)
matrix_b = np.random.rand(common_dim, cols_b)
expected = np.matmul(matrix_a, matrix_b)

def collapse(nested):
    for sub in nested:
        for item in sub:
            yield item

def cluster_by_key(data):
    groups = {}
    for k, v in data:
        groups[k] = groups.get(k, []) + [v]
    return groups.items()

def cluster_distributed(parts, part_func):
    global reducers_count
    buckets = [dict() for _ in range(reducers_count)]
    for part in parts:
        for k, v in part:
            bucket = buckets[part_func(k)]
            bucket[k] = bucket.get(k, []) + [v]
    return [(pid, sorted(b.items(), key=lambda x: x[0])) for pid, b in enumerate(buckets)]

def partition_func(obj):
    global reducers_count
    return hash(obj) % reducers_count

def run_distributed(source_fn, map_fn, reduce_fn, part_fn=partition_func, combine_fn=None):
    mapped = map(lambda reader: collapse(map(lambda kv: map_fn(*kv), reader)), source_fn())
    if combine_fn is not None:
        mapped = map(lambda part: collapse(map(lambda kv: combine_fn(*kv), cluster_by_key(part))), mapped)
    shuffled = cluster_distributed(mapped, part_fn)
    reduced = map(lambda rp: (rp[0], collapse(map(lambda group: reduce_fn(*group), rp[1]))), shuffled)
    print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (_,vs) in collapse([p for (_,p) in shuffled])])))
    return reduced

def to_matrix(output):
    out_list = list(output)
    if not out_list:
        return np.empty((rows_a, cols_b))
    max_i = max(i for ((i,k), _) in out_list) + 1
    max_k = max(k for ((i,k), _) in out_list) + 1
    result = np.empty((max_i, max_k))
    for (ik, val) in out_list:
        result[ik] = val
    return result

def input_source():
    batch_a = []
    for i in range(matrix_a.shape[0]):
        for j in range(matrix_a.shape[1]):
            batch_a.append(((0, i, j), matrix_a[i,j]))
    yield batch_a
    batch_b = []
    for j in range(matrix_b.shape[0]):
        for k in range(matrix_b.shape[1]):
            batch_b.append(((1, j, k), matrix_b[j,k]))
    yield batch_b

def map_join(key, val):
    src, idx1, idx2 = key
    if src == 0:
        yield (idx2, (src, idx1, val))
    else:
        yield (idx1, (src, idx2, val))

def reduce_join(key, vals):
    left = [v for v in vals if v[0] == 0]
    right = [v for v in vals if v[0] == 1]
    for l in left:
        for r in right:
            yield ((l[1], r[1]), l[2] * r[2])

def map_pass(key, val):
    yield (key, val)

def reduce_sum(key, vals):
    total = sum(vals)
    yield (key, total)

maps_count = 4
reducers_count = 2

joined_result = run_distributed(input_source, map_join, reduce_join, part_fn=partition_func, combine_fn=None)

joined_data = []
for pid, partition in joined_result:
    for item in partition:
        joined_data.append(item)

def source_joined():
    yield joined_data

mul_result = run_distributed(source_joined, map_pass, reduce_sum, part_fn=partition_func, combine_fn=None)

solution = []
for pid, partition in mul_result:
    for item in partition:
        solution.append(item)

print(np.allclose(expected, to_matrix(solution)))

340 key-value pairs were sent over a network.
1200 key-value pairs were sent over a network.
True


Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

будет работать при случайном подмножестве элементов, если все ненулевые элементы покрыты, нет дубликатов координат, и сохранены метки 'M'/'N' для различения матриц

In [49]:
rows_a, common_dim, cols_b = 4, 10, 30
matrix_a = np.random.rand(rows_a, common_dim)
matrix_b = np.random.rand(common_dim, cols_b)
expected = np.matmul(matrix_a, matrix_b)

def collapse(nested):
    for sub in nested:
        for item in sub:
            yield item

def cluster_by_key(data):
    groups = {}
    for k, v in data:
        groups[k] = groups.get(k, []) + [v]
    return groups.items()

def cluster_distributed(parts, part_func):
    global reducers_count
    buckets = [dict() for _ in range(reducers_count)]
    for part in parts:
        for k, v in part:
            bucket = buckets[part_func(k)]
            bucket[k] = bucket.get(k, []) + [v]
    return [(pid, sorted(b.items(), key=lambda x: x[0])) for pid, b in enumerate(buckets)]

def partition_func(obj):
    global reducers_count
    return hash(obj) % reducers_count

def run_distributed(source_fn, map_fn, reduce_fn, part_fn=partition_func, combine_fn=None):
    mapped = map(lambda reader: collapse(map(lambda kv: map_fn(*kv), reader)), source_fn())
    if combine_fn is not None:
        mapped = map(lambda part: collapse(map(lambda kv: combine_fn(*kv), cluster_by_key(part))), mapped)
    shuffled = cluster_distributed(mapped, part_fn)
    reduced = map(lambda rp: (rp[0], collapse(map(lambda group: reduce_fn(*group), rp[1]))), shuffled)
    print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (_,vs) in collapse([p for (_,p) in shuffled])])))
    return reduced

def to_matrix(output):
    out_list = list(output)
    if not out_list:
        return np.empty((rows_a, cols_b))
    max_i = max(i for ((i,k), _) in out_list) + 1
    max_k = max(k for ((i,k), _) in out_list) + 1
    result = np.empty((max_i, max_k))
    for (ik, val) in out_list:
        result[ik] = val
    return result

def input_source():
    batch_a = []
    for i in range(matrix_a.shape[0]):
        for j in range(matrix_a.shape[1]):
            batch_a.append(((0, i, j), matrix_a[i,j]))
    yield batch_a
    batch_b = []
    for j in range(matrix_b.shape[0]):
        for k in range(matrix_b.shape[1]):
            batch_b.append(((1, j, k), matrix_b[j,k]))
    yield batch_b

def map_join(key, val):
    src, idx1, idx2 = key
    if src == 0:
        yield (idx2, (src, idx1, val))
    else:
        yield (idx1, (src, idx2, val))

def reduce_join(key, vals):
    left = [v for v in vals if v[0] == 0]
    right = [v for v in vals if v[0] == 1]
    for l in left:
        for r in right:
            yield ((l[1], r[1]), l[2] * r[2])

def map_pass(key, val):
    yield (key, val)

def reduce_sum(key, vals):
    total = sum(vals)
    yield (key, total)

maps_count = 4
reducers_count = 2

joined_result = run_distributed(input_source, map_join, reduce_join, part_fn=partition_func, combine_fn=None)

joined_data = []
for pid, partition in joined_result:
    for item in partition:
        joined_data.append(item)

def source_joined():
    yield joined_data

mul_result = run_distributed(source_joined, map_pass, reduce_sum, part_fn=partition_func, combine_fn=None)

solution = []
for pid, partition in mul_result:
    for item in partition:
        solution.append(item)

print(np.allclose(expected, to_matrix(solution)))

340 key-value pairs were sent over a network.
1200 key-value pairs were sent over a network.
True
